# 4. 평가 및 시각화

RandomForest, LightGBM, XGBoost, AutoGluon Best Model, Ensemble 5개 모델을 비교합니다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
import lightgbm as lgb
import xgboost as xgb
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

FIGURE_DIR = '../outputs/figures'
os.makedirs(FIGURE_DIR, exist_ok=True)

train_data = pd.read_csv('../data/train_processed.csv')
test_raw   = pd.read_csv('../data/test_processed.csv')
test_ids   = pd.read_csv('../data/test_ids.csv')['ID']
X = train_data.drop('label', axis=1)
y = train_data['label']

# AutoGluon 로드
predictor = TabularPredictor.load('../outputs/ag_models')
ag_best_name = predictor.get_model_best()
print(f'AutoGluon Best Model: {ag_best_name}')

## 4.1 모델별 성능 비교

5-Fold CV 기반으로 RF / LightGBM / XGBoost 성능을 측정하고,
AutoGluon best model과 3개 모델 Soft Voting 앙상블을 함께 비교합니다.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

sklearn_models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'LightGBM':     lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1),
    'XGBoost':      xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, random_state=42, eval_metric='logloss'),
}

# 각 fold의 예측 확률 저장 (앙상블용)
fold_probas = {name: [] for name in sklearn_models}
fold_true   = []
fold_indices = []

eval_results = []

for name, model in sklearn_models.items():
    metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
    for tr_idx, val_idx in skf.split(X, y):
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred  = model.predict(X.iloc[val_idx])
        proba = model.predict_proba(X.iloc[val_idx])[:, 1]
        metrics['accuracy'].append(accuracy_score(y.iloc[val_idx], pred))
        metrics['precision'].append(precision_score(y.iloc[val_idx], pred))
        metrics['recall'].append(recall_score(y.iloc[val_idx], pred))
        metrics['f1'].append(f1_score(y.iloc[val_idx], pred))
        if name == 'LightGBM':  # fold 인덱스는 한 번만 저장
            fold_indices.append(val_idx)
            fold_true.append(y.iloc[val_idx].values)
        fold_probas[name].append(proba)

    eval_results.append({
        'Model':     name,
        'Accuracy':  np.mean(metrics['accuracy']),
        'Precision': np.mean(metrics['precision']),
        'Recall':    np.mean(metrics['recall']),
        'F1-score':  np.mean(metrics['f1']),
    })
    print(f"{name}: Acc={np.mean(metrics['accuracy']):.4f}, F1={np.mean(metrics['f1']):.4f}")

In [ ]:
# AutoGluon best model - fold별 성능
ag_metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
ag_fold_probas = []

for tr_idx, val_idx in skf.split(X, y):
    val_X = X.iloc[val_idx]
    pred  = predictor.predict(val_X, model=ag_best_name)
    proba = predictor.predict_proba(val_X, model=ag_best_name).iloc[:, 1].values
    ag_metrics['accuracy'].append(accuracy_score(y.iloc[val_idx], pred))
    ag_metrics['precision'].append(precision_score(y.iloc[val_idx], pred))
    ag_metrics['recall'].append(recall_score(y.iloc[val_idx], pred))
    ag_metrics['f1'].append(f1_score(y.iloc[val_idx], pred))
    ag_fold_probas.append(proba)

eval_results.append({
    'Model':     'AutoGluon Best',
    'Accuracy':  np.mean(ag_metrics['accuracy']),
    'Precision': np.mean(ag_metrics['precision']),
    'Recall':    np.mean(ag_metrics['recall']),
    'F1-score':  np.mean(ag_metrics['f1']),
})
print(f"AutoGluon Best: Acc={np.mean(ag_metrics['accuracy']):.4f}, F1={np.mean(ag_metrics['f1']):.4f}")

In [ ]:
# Soft Voting Ensemble (RF + LightGBM + XGBoost)
ens_metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
ens_fold_probas = []

for i, val_idx in enumerate(fold_indices):
    avg_proba = np.mean([fold_probas[n][i] for n in sklearn_models], axis=0)
    pred = (avg_proba >= 0.5).astype(int)
    ens_metrics['accuracy'].append(accuracy_score(fold_true[i], pred))
    ens_metrics['precision'].append(precision_score(fold_true[i], pred))
    ens_metrics['recall'].append(recall_score(fold_true[i], pred))
    ens_metrics['f1'].append(f1_score(fold_true[i], pred))
    ens_fold_probas.append(avg_proba)

eval_results.append({
    'Model':     'Ensemble (RF+LGBM+XGB)',
    'Accuracy':  np.mean(ens_metrics['accuracy']),
    'Precision': np.mean(ens_metrics['precision']),
    'Recall':    np.mean(ens_metrics['recall']),
    'F1-score':  np.mean(ens_metrics['f1']),
})
print(f"Ensemble: Acc={np.mean(ens_metrics['accuracy']):.4f}, F1={np.mean(ens_metrics['f1']):.4f}")

eval_df = pd.DataFrame(eval_results).set_index('Model')
print('\n── 전체 성능 비교 ──')
print(eval_df.round(4))

## 4.2 성능 비교 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
eval_df.plot(kind='bar', ax=ax, rot=15)
ax.set_title('모델별 성능 비교')
ax.set_ylabel('Score')
ax.set_ylim(0.5, 1.0)
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(f'{FIGURE_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장 완료: {FIGURE_DIR}/model_comparison.png')

## 4.3 Confusion Matrix (전체 모델)

In [ ]:
# 마지막 fold 기준으로 각 모델 confusion matrix 시각화
last_val_idx = fold_indices[-1]
y_true_last  = fold_true[-1]

cm_models = {}
for name, model in sklearn_models.items():
    model.fit(X.iloc[fold_indices[-1]], y.iloc[fold_indices[-1]])  # 마지막 fold train으로 재학습
    # 실제로는 fold_probas 마지막 값 사용
    pred = (fold_probas[name][-1] >= 0.5).astype(int)
    cm_models[name] = confusion_matrix(y_true_last, pred)

# AutoGluon
ag_pred_last = (ag_fold_probas[-1] >= 0.5).astype(int)
cm_models['AutoGluon Best'] = confusion_matrix(y_true_last, ag_pred_last)

# Ensemble
ens_pred_last = (ens_fold_probas[-1] >= 0.5).astype(int)
cm_models['Ensemble'] = confusion_matrix(y_true_last, ens_pred_last)

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, cm) in zip(axes, cm_models.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Smoker', 'Smoker'],
                yticklabels=['Non-Smoker', 'Smoker'], ax=ax)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
fig.tight_layout()
fig.savefig(f'{FIGURE_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장 완료: {FIGURE_DIR}/confusion_matrix.png')

## 4.4 ROC Curve (전체 모델)

In [ ]:
# 전체 fold 확률을 이어붙여 ROC 계산
y_true_all = np.concatenate(fold_true)

roc_data = {}
for name in sklearn_models:
    proba_all = np.concatenate(fold_probas[name])
    fpr, tpr, _ = roc_curve(y_true_all, proba_all)
    roc_data[name] = (fpr, tpr, auc(fpr, tpr))

ag_proba_all  = np.concatenate(ag_fold_probas)
fpr, tpr, _   = roc_curve(y_true_all, ag_proba_all)
roc_data['AutoGluon Best'] = (fpr, tpr, auc(fpr, tpr))

ens_proba_all = np.concatenate(ens_fold_probas)
fpr, tpr, _   = roc_curve(y_true_all, ens_proba_all)
roc_data['Ensemble (RF+LGBM+XGB)'] = (fpr, tpr, auc(fpr, tpr))

fig, ax = plt.subplots(figsize=(9, 7))
for name, (fpr, tpr, roc_auc) in roc_data.items():
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve 비교')
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(f'{FIGURE_DIR}/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장 완료: {FIGURE_DIR}/roc_curve.png')

## 4.5 Feature Importance (AutoGluon Best Model)

In [ ]:
feat_imp = predictor.feature_importance(train_data)
top20 = feat_imp.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
top20['importance'][::-1].plot(kind='barh', color='steelblue', ax=ax)
ax.set_yticklabels(top20.index[::-1])
ax.set_title(f'Feature Importance Top 20 ({ag_best_name})')
ax.set_xlabel('Importance')
fig.tight_layout()
fig.savefig(f'{FIGURE_DIR}/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장 완료: {FIGURE_DIR}/feature_importance.png')

## 4.6 결과 요약

In [ ]:
best_row = eval_df['F1-score'].idxmax()

print('=' * 55)
print('📊 최종 결과 요약')
print('=' * 55)
print(eval_df.round(4).to_string())
print('=' * 55)
print(f'\n🏆 Best Model (F1 기준): {best_row}')
print(f'   Accuracy:  {eval_df.loc[best_row, "Accuracy"]:.4f}')
print(f'   Precision: {eval_df.loc[best_row, "Precision"]:.4f}')
print(f'   Recall:    {eval_df.loc[best_row, "Recall"]:.4f}')
print(f'   F1-score:  {eval_df.loc[best_row, "F1-score"]:.4f}')
print('\n저장된 시각화 파일:')
for f in os.listdir(FIGURE_DIR):
    if f.endswith('.png'):
        print(f'  📈 {FIGURE_DIR}/{f}')